# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

**Group / repo name:** `aidams-lab1-<surname1>-<surname2>-...`  
**Submitter (one person):**  
**Repo URL:**  
**Streamlit Cloud URL (bonus):**  

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [2]:
# Import required libraries
# - pandas for data manipulation
# - numpy for numerical operations
# - plotly.express and plotly.graph_objects for interactive visualizations
# - Any other libraries you need

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [10]:
# Load the steel plants dataset
# Tip: start with df.columns / df.head() and adapt names if your file differs slightly
#
# Common columns in this steel plant dataset:
# - Plant name (English)
# - Owner
# - Country/Area, Region
# - Coordinates  (often a single "lat, lon" string — not separate latitude/longitude columns)
# - Plant age (years)
# - Capacity field: usually "Nominal crude steel capacity (ttpa)" (check your df.columns to confirm the exact name)
#   (Older datasets may also include additional fields like ferronickel/sinter/coking/pelletizing capacities,
#    but most analyses focus on nominal crude steel capacity as the main output.)

plant_data = pd.read_csv("plant_data.csv")
plant_capacities = pd.read_csv("plant_capacities.csv")
plant_production = pd.read_csv("plant_production.csv")

plant_data.columns
plant_data.head()



,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Steel products,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,"billet, wire rod, angle, flat, bar, square bar...",unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,"billet, wire rod, rebar",building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,"wire rod, rebar, bar, billet, round bar, wire",unknown,4500,2025-10-06,unknown,no,EAF,unknown,NaN,unknown
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"billet, rebar","building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,"pipe, tube, flat","automotive, building and infrastructure, energ...",11000,2025-04-30,2025-12-04,no,DRI; EAF; BF; BOF,unknown,unknown,unknown


In [11]:
plant_capacities.head()

,GEM plant ID,Plant name (English),Plant name (other language),Country/area,Main production equipment,Status,Start date,Nominal crude steel capacity (ttpa),Nominal BOF steel capacity (ttpa),Nominal EAF steel capacity (ttpa),Nominal IF steel capacity (ttpa),Other/unspecified steel capacity (ttpa),Nominal iron capacity (ttpa),Nominal BF capacity (ttpa),Nominal DRI capacity (ttpa),Other/unspecified iron capacity (ttpa)
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,Türkiye,EAF,operating,1983,1100,NaN,1100,NaN,NaN,NaN,NaN,NaN,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Namibia,EAF,construction,unknown,3000,NaN,3000,NaN,NaN,NaN,NaN,NaN,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,Russia,EAF,operating,2014,1600,NaN,1600,NaN,NaN,NaN,NaN,NaN,NaN
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,Bangladesh,EAF,operating,2015,1400,NaN,1400,NaN,NaN,NaN,NaN,NaN,NaN
4,P100000120284,Tianjin New Tiangang United Special Steel Co Ltd,天津天钢联合特钢有限公司，天津天钢联合钢铁有限公司,China,BF,retired,2009-06-24,NaN,NaN,NaN,NaN,NaN,2360,2360,NaN,NaN


In [23]:
plant_production.head()

,GEM plant ID,Plant name (English),Type of production,2019,2020,2021,2022,2023,2024,2025
0,P100000120802,Abinsk Electric Steel Works,Crude steel production (ttpa),1500,1400,1600,unknown,unknown,unknown,unknown
1,P100000120802,Abinsk Electric Steel Works,EAF steel production (ttpa),1500,1400,1600,unknown,unknown,unknown,unknown
2,P100000120620,Acciaierie d'Italia Taranto steel plant,Crude steel production (ttpa),4300,3400,4100,3800,3000,2000,unknown
3,P100000120620,Acciaierie d'Italia Taranto steel plant,BOF steel production (ttpa),4300,3400,4100,3800,3000,2000,unknown
4,P100000120634,Acciaierie Venete Borgo Valsugana steel plant,Crude steel production (ttpa),488,475,629,unknown,unknown,unknown,unknown


---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [13]:
# Display dataset shape
plant_data.shape

(1293, 44)

In [14]:
# Display column information and data types
# Start here: print(df.columns) and adapt column names in later cells if needed
plant_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1293 entries, 0 to 1292
Data columns (total 44 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   GEM plant ID                        1293 non-null   object
 1   Plant name (English)                1293 non-null   object
 2   Plant name (other language)         791 non-null    object
 3   Other plant names (English)         742 non-null    object
 4   Other plant names (other language)  335 non-null    object
 5   Owner                               1293 non-null   object
 6   Owner (other language)              578 non-null    object
 7   Owner GEM entity ID                 1293 non-null   object
 8   Owner PermID                        1293 non-null   object
 9   SOE status                          212 non-null    object
 10  Parent (English)                    1293 non-null   object
 11  Parent GEM entity ID                1293 non-null   obje

In [15]:
# Check for missing values
plant_data.isnull().sum()


,0
GEM plant ID,0
Plant name (English),0
Plant name (other language),502
Other plant names (English),551
Other plant names (other language),958
Owner,0
Owner (other language),715
Owner GEM entity ID,0
Owner PermID,0
SOE status,1081


### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [22]:
# Display descriptive statistics
# using nominal crude steel capacity
plant_capacities["Nominal crude steel capacity (ttpa)"] = pd.to_numeric(
    plant_capacities["Nominal crude steel capacity (ttpa)"],
    errors="coerce" # otherwise code NaN can be considered as error
)

average_capacity = plant_capacities[
    "Nominal crude steel capacity (ttpa)"
].mean()

average_capacity


np.float64(2325.521215959468)

In [24]:
plant_data["Coordinates"].head()

,Coordinates
0,"36.7474130, 36.2173300"
1,"-17.3978660, 15.8910220"
2,"44.8819380, 38.1275100"
3,"22.4720080, 91.7348270"
4,"40.5089930, 17.2075890"


In [52]:
plant_data[['Latitude', 'Longitude']] = plant_data['Coordinates'].str.split(',', expand=True)

plant_data['Latitude'] = pd.to_numeric(plant_data['Latitude'])
plant_data['Longitude'] = pd.to_numeric(plant_data['Longitude'])

print("Latitude Range:")
print(f"Min: {plant_data['Latitude'].min()}, Max: {plant_data['Latitude'].max()}")

print("\nLongitude Range:")
print(f"Min: {plant_data['Longitude'].min()}, Max: {plant_data['Longitude'].max()}")

Latitude Range:
Min: -37.831379, Max: 67.189096

Longitude Range:
Min: -123.163599, Max: 174.728098


In [29]:
plant_data["Plant age"].head()

,Plant age
0,43
1,1
2,12
3,11
4,61


In [30]:
plant_data["Plant age"].describe()

,Plant age
count,1232
unique,144
top,unknown
freq,107


In [31]:
fig = px.histogram(
    plant_data,
    x="Plant age",
    nbins=30,
    title="Distribution of Steel Plant Age"
)

fig.show()

### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [34]:
# Count plants by country/region
plant_capacities["Country/area"].value_counts()


,count
Country/area,
China,700
India,196
United States,99
Iran,85
Japan,61
...,...
Hong Kong,1
Slovenia,1
Singapore,1


In [35]:
# Count plants by Owner (company)
plant_data["Owner"].value_counts()

,count
Owner,
Nucor Corp,13
Cleveland-Cliffs Inc,12
Nippon Steel Corp,10
Steel Authority of India Ltd,8
Gerdau Ameristeel Corp,8
...,...
Hebei Huaxi Special Steel Co Ltd,1
Hebei Huaxin Special Steel Co Ltd,1
Hebei Jingdong Pipe Industry Co Ltd,1


### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?


In [36]:
# Calculate total capacity
capacity = "Nominal crude steel capacity (ttpa)"

plant_capacities[capacity] = pd.to_numeric(
    plant_capacities[capacity], errors="coerce"
)
plant_capacities[capacity].sum()

np.float64(3671998.0)

In [41]:
# Group by Owner and sum capacity
capacity_df = plant_capacities.merge(
    plant_data[["GEM plant ID", "Owner"]],
    on="GEM plant ID"
)

capacity_df.groupby("Owner")[capacity].sum().sort_values(ascending=False).head(10)

,Nominal crude steel capacity (ttpa)
Owner,
ArcelorMittal Nippon Steel India Ltd,86500.0
JSW Steel Ltd,59859.0
Steel Authority of India Ltd,57190.0
Nippon Steel Corp,53666.0
POSCO Holdings Inc,46757.0
JFE Steel Corp,34843.0
Tata Steel Ltd,34830.0
Angang Steel Co Ltd,33700.0
Jindal Steel Limited Ltd,33200.0


---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [45]:
# Create a scatter_geo or scatter_mapbox plot
# Hint: Use plotly.express.scatter_geo() or scatter_mapbox()
#
# Coordinates hint:
#   The dataset usually stores location in a single Coordinates column like "lat, lon".
#   Split it before plotting

plant_data["Latitude"] = plant_data["Coordinates"].str.split(",").str[0]
plant_data["Longitude"] = plant_data["Coordinates"].str.split(",").str[1]

map_df = plant_data.merge(
    plant_capacities[["GEM plant ID", "Nominal crude steel capacity (ttpa)"]],
    on="GEM plant ID",
    how="left"
)

fig = px.scatter_geo(
    map_df,
    lat="Latitude",
    lon="Longitude",
    color="Country/area",
    hover_name="Plant name (English)",
    hover_data=["Owner", "Nominal crude steel capacity (ttpa)"]
)

fig.show()

### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [49]:
# Create scatter map with size parameter based on capacity

map_df = map_df.dropna(subset=["Nominal crude steel capacity (ttpa)"])

fig = px.scatter_geo(
    map_df,
    lat="Latitude",
    lon="Longitude",
    size="Nominal crude steel capacity (ttpa)",
    color="Owner",
    hover_name="Plant name (English)",
    hover_data=["Country/area", "Nominal crude steel capacity (ttpa)"],
    title="Global Steel Plants by Capacity"
)

fig.show()

### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [54]:
# Create density heatmap
# Hint: Use plotly.express.density_mapbox()
density_df = plant_data.dropna(subset=["Latitude", "Longitude"])
fig = px.density_mapbox(
    density_df,
    lat="Latitude",
    lon="Longitude",
    radius=10,
    zoom=1,
    mapbox_style="carto-positron",
    title="Global Steel Plant Density"
)

fig.show()


---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.


### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [55]:
# Load LitPop sample (exposure / population–asset data)

litpop_china = pd.read_hdf(
    "LitPop_pc_300_arcsec_CHN_v1.hdf5"
)
litpop_china.head()


,value,latitude,longitude,geometry,region_id,impf_
0,5.280440e+09,20.041667,110.208333,POINT (110.20833333 20.04166667),156,1
1,4.040559e+07,20.041667,110.625000,POINT (110.625 20.04166667),156,1
2,4.190224e+07,20.041667,110.708333,POINT (110.70833333 20.04166667),156,1
3,8.813872e+07,19.958333,109.541667,POINT (109.54166667 19.95833333),156,1
4,1.879947e+08,19.958333,109.625000,POINT (109.625 19.95833333),156,1


In [56]:
# Inspect LitPop data (columns, dtypes, missing values, value ranges)

print("Columns:")
print(litpop_china.columns)

print("\nData types:")
print(litpop_china.dtypes)

print("\nMissing values:")
print(litpop_china.isnull().sum())

print("\nValue ranges:")
print(litpop_china.describe())


Columns:
Index(['value', 'latitude', 'longitude', 'geometry', 'region_id', 'impf_'], dtype='object')

Data types:
value        float64
latitude     float64
longitude    float64
geometry      object
region_id      int64
impf_          int64
dtype: object

Missing values:
value        0
latitude     0
longitude    0
geometry     0
region_id    0
impf_        0
dtype: int64

Value ranges:
              value       latitude      longitude  region_id     impf_
count  1.369910e+05  136991.000000  136991.000000   136991.0  136991.0
mean   2.870462e+08      36.557225     103.832300      156.0       1.0
std    3.697379e+09       7.145996      14.370976        0.0       0.0
min    0.000000e+00      18.208333      73.625000      156.0       1.0
25%    2.016376e+04      31.125000      91.208333      156.0       1.0
50%    3.790894e+05      36.708333     104.708333      156.0       1.0
75%    4.678025e+06      41.875000     115.708333      156.0       1.0
max    2.873477e+11      53.541667     134.

### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [57]:
# Calculate distances or perform spatial join
# Hint: You might calculate haversine distance or use a spatial library
from scipy.spatial import cKDTree

litpop_coords = litpop_china[['latitude', 'longitude']].to_numpy()

plant_coords = plant_data[['Latitude', 'Longitude']].to_numpy()

tree = cKDTree(litpop_coords)

distances, indices = tree.query(plant_coords)

print(indices[:5])
print(distances[:5])


[ 52612 101818  52059 120459  52059]
[37.4885303  79.54345788 35.93498198  5.32150513 56.43054268]


In [65]:
# Merge datasets

from sklearn.neighbors import BallTree

# Load the three LitPop datasets
litpop_china = pd.read_hdf("LitPop_pc_300_arcsec_CHN_v1.hdf5")
litpop_india = pd.read_hdf("LitPop_pc_300_arcsec_IND_v1.hdf5")
litpop_japan = pd.read_hdf("LitPop_pc_300_arcsec_JPN_v1.hdf5")

# Add country labels
litpop_china["Country/area"] = "China"
litpop_india["Country/area"] = "India"
litpop_japan["Country/area"] = "Japan"

# Keep ONLY steel plants in these three countries
plants_3countries = plant_data[
    plant_data["Country/area"].isin(["China", "India", "Japan"])
].copy()

print(plants_3countries["Country/area"].value_counts())

Country/area
China    458
India    113
Japan     42
Name: count, dtype: int64


In [68]:
# Match each steel plant to the nearest LitPop grid cell

litpop_dict = {
    "China": litpop_china,
    "India": litpop_india,
    "Japan": litpop_japan
}

matched_parts = []

for country in ["China", "India", "Japan"]:

    plants_country = plants_3countries[
        plants_3countries["Country/area"] == country
    ].copy().reset_index(drop=True)

    litpop_country = litpop_dict[country].reset_index(drop=True)

    # Convert latitude and longitude to radians
    plant_coords = np.radians(
        plants_country[["Latitude", "Longitude"]].to_numpy()
    )

    litpop_coords = np.radians(
        litpop_country[["latitude", "longitude"]].to_numpy()
    )

    # Build nearest-neighbor model
    tree = BallTree(litpop_coords, metric="haversine")

    distances, indices = tree.query(plant_coords, k=1)

    # Get nearest LitPop cells
    nearest = litpop_country.iloc[
        indices[:, 0]
    ].reset_index(drop=True)

    # Attach LitPop exposure data
    plants_country["LitPop_value"] = nearest["value"]
    plants_country["LitPop_latitude"] = nearest["latitude"]
    plants_country["LitPop_longitude"] = nearest["longitude"]

    # Distance in kilometers
    plants_country["distance_to_LitPop_km"] = (
        distances[:, 0] * 6371
    )

    matched_parts.append(plants_country)

plant_exposure = pd.concat(
    matched_parts,
    ignore_index=True
)

plant_exposure.head()

print("Total matched plants:", len(plant_exposure))

print("\nCountries:")
print(plant_exposure["Country/area"].value_counts())

print("\nMatching distance (km):")
print(plant_exposure["distance_to_LitPop_km"].describe())

Total matched plants: 613

Countries:
Country/area
China    458
India    113
Japan     42
Name: count, dtype: int64

Matching distance (km):
count    613.000000
mean       3.440857
std        1.616343
min        0.031360
25%        2.327039
50%        3.526021
75%        4.383673
max       18.271759
Name: distance_to_LitPop_km, dtype: float64


### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [70]:
# Create visualization of plants colored by LitPop exposure metrics
import plotly.express as px

fig = px.scatter_map(
    plant_exposure,
    lat="Latitude",
    lon="Longitude",

    color="LitPop_value",

    hover_name="Plant name (English)",
    hover_data={
        "Owner": True,
        "Country/area": True,
        "LitPop_value": ":.2e",
        "distance_to_LitPop_km": ":.2f",
        "Latitude": False,
        "Longitude": False
    },

    map_style="carto-positron",
    zoom=2,

    title="Steel Plants and Local LitPop Exposure"
)

fig.update_traces(
    marker={"size": 10}
)
fig.update_layout(
    height=700,
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)
fig.show()

---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [83]:
capacity_col = "Nominal crude steel capacity (ttpa)"

plant_capacities[capacity_col] = pd.to_numeric(
    plant_capacities[capacity_col],
    errors="coerce"
)

plant_exposure = plant_exposure.drop(
    columns=[capacity_col],
    errors="ignore"
)

plant_exposure = plant_exposure.merge(
    plant_capacities[
        ["GEM plant ID", capacity_col]
    ],
    on="GEM plant ID",
    how="left"
)

company_summary = (
    plant_exposure
    .groupby("Owner")
    .agg(
        total_capacity=(capacity_col, "sum"),
        number_of_plants=("GEM plant ID", "nunique"),
        average_LitPop_exposure=("LitPop_value", "mean"),
        geographic_spread=("Country/area", "nunique")
    )
    .reset_index()
)

company_summary = company_summary.sort_values(
    by="total_capacity",
    ascending=False
)

print("Company-level aggregated metrics:")
company_summary.head(10)

Company-level aggregated metrics:


,Owner,total_capacity,number_of_plants,average_LitPop_exposure,geographic_spread
344,Rizhao Steel Holding Group Co Ltd,373600.0,1,7.849491e+08,1
409,Steel Authority of India Ltd,362952.0,8,3.100962e+09,1
16,ArcelorMittal Nippon Steel India Ltd,358000.0,5,2.571208e+09,1
299,Maanshan Iron & Steel Co Ltd,332800.0,1,1.416193e+10,1
235,Jiangsu Shagang Iron & Steel Co Ltd,329360.0,1,5.778523e+09,1
219,JSW Steel Ltd,288651.0,5,3.176233e+08,1
275,Lianfeng Steel (Zhangjiagang) Co Ltd,270000.0,1,3.289812e+09,1
36,Bengang Steel Plates Co Ltd,239040.0,1,1.030251e+10,1
4,Angang Steel Co Ltd,226500.0,3,4.168825e+10,1
320,Nippon Steel Corp,198179.0,10,1.518464e+10,1


### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [84]:
# Calculate company representative locations

company_locations = (
    plant_exposure
    .groupby("Owner")
    .agg(
        company_latitude=("Latitude", "mean"),
        company_longitude=("Longitude", "mean")
    )
    .reset_index()
)

company_data = company_summary.merge(
    company_locations,
    on="Owner",
    how="left"
)

company_data.head(10)

,Owner,total_capacity,number_of_plants,average_LitPop_exposure,geographic_spread,company_latitude,company_longitude
0,Rizhao Steel Holding Group Co Ltd,373600.0,1,7.849491e+08,1,35.165350,119.366672
1,Steel Authority of India Ltd,362952.0,8,3.100962e+09,1,22.534338,84.881886
2,ArcelorMittal Nippon Steel India Ltd,358000.0,5,2.571208e+09,1,20.275123,75.689233
3,Maanshan Iron & Steel Co Ltd,332800.0,1,1.416193e+10,1,31.698884,118.468194
4,Jiangsu Shagang Iron & Steel Co Ltd,329360.0,1,5.778523e+09,1,31.983146,120.638992
5,JSW Steel Ltd,288651.0,5,3.176233e+08,1,16.987253,75.154574
6,Lianfeng Steel (Zhangjiagang) Co Ltd,270000.0,1,3.289812e+09,1,31.855205,120.730505
7,Bengang Steel Plates Co Ltd,239040.0,1,1.030251e+10,1,41.274132,123.722099
8,Angang Steel Co Ltd,226500.0,3,4.168825e+10,1,41.132436,122.863560
9,Nippon Steel Corp,198179.0,10,1.518464e+10,1,34.874428,135.366305


### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [85]:
# Create company-level visualization

import plotly.express as px

company_map_data = company_data.dropna(
    subset=[
        "company_latitude",
        "company_longitude",
        "total_capacity",
        "average_LitPop_exposure"
    ]
).copy()

fig = px.scatter_map(
    company_map_data,

    lat="company_latitude",
    lon="company_longitude",
    size="total_capacity",
    color="average_LitPop_exposure",
    hover_name="Owner",
    hover_data={
        "total_capacity": ":,.0f",
        "number_of_plants": True,
        "average_LitPop_exposure": ":.2e",
        "geographic_spread": True,
        "company_latitude": False,
        "company_longitude": False
    },

    map_style="carto-positron",
    zoom=2,

    title="Steel Companies: Capacity and LitPop Exposure"
)

fig.update_layout(
    height=700,
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading


In [86]:
# Save processed datasets

plant_data.to_csv(
    "cleaned_plant_data.csv",
    index=False
)

plant_exposure.to_csv(
    "plant_exposure.csv",
    index=False
)

company_data.to_csv(
    "company_summary.csv",
    index=False
)

print("Datasets saved successfully!")

print("cleaned_plant_data.csv:", plant_data.shape)
print("plant_exposure.csv:", plant_exposure.shape)
print("company_summary.csv:", company_data.shape)


Datasets saved successfully!
cleaned_plant_data.csv: (1293, 46)
plant_exposure.csv: (5151, 54)
company_summary.csv: (535, 7)


### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

In [87]:
# This cell is for notes/observations about your dashboard
# What works well?
# What could be improved?
# Any performance issues with large datasets?

# Dashboard observations

print("""
Dashboard observations:

- The dashboard allows users to compare steel plant capacity and local
  LitPop exposure across China, India, and Japan.

- Interactive filters for company, country, and capacity make it easier
  to explore differences between plants and companies.

- Marker size represents steel capacity, while marker color represents
  local LitPop exposure.

- The current LitPop analysis is limited to China, India, and Japan,
  because only these three LitPop samples were used.

- Company locations represent the centroid of plant locations rather
  than actual corporate headquarters.

- For larger global datasets, Parquet and cached data loading could
  improve dashboard performance.
""")


Dashboard observations:

- The dashboard allows users to compare steel plant capacity and local
  LitPop exposure across China, India, and Japan.

- Interactive filters for company, country, and capacity make it easier
  to explore differences between plants and companies.

- Marker size represents steel capacity, while marker color represents
  local LitPop exposure.

- The current LitPop analysis is limited to China, India, and Japan,
  because only these three LitPop samples were used.

- Company locations represent the centroid of plant locations rather
  than actual corporate headquarters.

- For larger global datasets, Parquet and cached data loading could
  improve dashboard performance.



---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
